# 00 — Coleta de Dados

Download dos dados brutos: arquivos SCR anuais (BCB) e variáveis macroeconômicas (SGS/BCB).

> **Pré-requisito**: internet. Executar apenas uma vez — dados já existem em `data/raw/`.

## 1. Download dos arquivos SCR (Banco Central)

Baixa os arquivos ZIP anuais do SCR (Sistema de Informações de Crédito) diretamente da API pública do BCB. Os arquivos são salvos em `data/raw/scr/`.

**Executar apenas uma vez** — os arquivos somam ~3.8 GB.

In [ ]:
import requests
from pathlib import Path

# Caminho ancorado na RAIZ do projeto. Antes era relativo ao diretorio de trabalho,
# o que criava notebooks/data/raw/scr quando executado a partir de notebooks/.
BASE = Path().resolve().parent
RAW  = BASE / "data" / "raw" / "scr"
RAW.mkdir(parents=True, exist_ok=True)

for ano in range(2012, 2026):
    destino = RAW / f"scr_{ano}.zip"
    if destino.exists():
        print(f"{ano}: ja existe ({destino.stat().st_size/1e6:.0f} MB), pulando")
        continue

    url = f"https://www.bcb.gov.br/pda/desig/planilha_{ano}.zip"
    print(f"{ano}: baixando...", end=" ", flush=True)
    try:
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        destino.write_bytes(r.content)
        print(f"ok ({len(r.content)/1e6:.1f} MB)")
    except requests.HTTPError as e:
        print(f"erro: {e}")

print(f"\nArquivos ZIP em {RAW.name}/: {len(list(RAW.glob('*.zip')))}")

## 2. Download das variáveis macroeconômicas (API SGS/BCB)

Coleta as séries mensais usadas no pipeline via API do Sistema Gerenciador de Séries Temporais
(SGS) do Banco Central.

| Código | Identificador | Conteúdo | Uso no modelo |
|---|---|---|---|
| 4189 | `selic_meta` | Meta Selic (Copom), % a.a. | nível (taxa) |
| 433 | `ipca` | IPCA, variação mensal | nível (já é variação) |
| 24369 | `desemprego` | Taxa de desocupação, PNAD Contínua | nível (taxa) |
| 20783 | `spread_pj` | Spread médio das operações de crédito PJ, p.p. | nível (taxa) — **novo** |
| 4394 | `confianca_empresarial` | Índice de confiança empresarial (FGV) | nível e variação |
| 24364 | `ibcbr_dessaz` | IBC-Br dessazonalizado | **variação de 12 meses** — **novo** |
| 20539 | `credito_saldo_sfn` | Saldo da carteira de crédito total, R$ mi | **variação de 12 meses** |
| 4380 | `pib_nominal` | PIB mensal a valores correntes, R$ mi | não entra |
| 20541 | `credito_pj_saldo` | Saldo da carteira de crédito PJ, R$ mi | não entra |

> **Revisão A4c.** Versões anteriores chamavam de `spread_pj` o SGS 20539 — um *saldo* de crédito,
> não um spread — e de `pib_var` o nível nominal do PIB. Os identificadores foram corrigidos
> (`credito_saldo_sfn`, `pib_nominal`), o spread PJ de fato (20783) e o IBC-Br (24364) foram
> acrescentados, e níveis nominais passam a entrar no modelo apenas em variação.

> Versões anteriores deste notebook usavam códigos inválidos (1, 28 e 4825), que retornam erro
> 406/404 na API, e coletavam a cotação do câmbio — que não é utilizada pelo modelo e foi removida.

In [ ]:
import pandas as pd
import requests
from pathlib import Path

BASE = Path().resolve().parent
PROC = BASE / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

def buscar_sgs(codigo, nome, data_inicio="01/01/2012"):
    """Baixa uma serie do Sistema Gerenciador de Series Temporais (SGS/BCB)."""
    url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}"
           f"/dados?formato=json&dataInicial={data_inicio}")
    r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df["data"]  = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
    return df.rename(columns={"valor": nome})[["data", nome]]

# ---------------------------------------------------------------------------------------
# Revisao A4c — identificadores corrigidos e series novas.
#
# Historico: as versoes anteriores chamavam de "spread_pj" o SGS 20539 — que e um SALDO de
# credito em R$ milhoes, nao um spread — e de "pib_var" o SGS 4380, que e o NIVEL nominal do
# PIB mensal. Os nomes foram corrigidos para refletir o conteudo, e duas series foram
# acrescentadas: o spread medio PJ de fato (20783) e o IBC-Br dessazonalizado (24364), que o
# notebook 03 converte em variacao. Niveis nominais nao entram no modelo em nivel.
#
# Defasagem de publicacao (revisao A1), verificada na API em 10/11/2026 e parametrizada no
# notebook 03: Selic 0 | IPCA 1 | desemprego 2 | confianca 1 | spread e saldos 2 | IBC-Br 2.
# ---------------------------------------------------------------------------------------
SERIES = {
    # taxas e variacoes — podem entrar no modelo como estao
    4189:  "selic_meta",             # Meta Selic definida pelo Copom (% a.a.)
    433:   "ipca",                   # IPCA, variacao mensal (%)
    24369: "desemprego",             # Taxa de desocupacao, PNAD Continua (%)
    20783: "spread_pj",              # Spread medio das operacoes de credito - PJ (p.p.)   [novo]
    4394:  "confianca_empresarial",  # Indice de confianca empresarial, sondagem FGV
    # indices e niveis nominais — o notebook 03 usa apenas a VARIACAO destes
    24364: "ibcbr_dessaz",           # IBC-Br dessazonalizado (indice de atividade)         [novo]
    4380:  "pib_nominal",            # PIB mensal a valores correntes, R$ milhoes (ex-"pib_var")
    20539: "credito_saldo_sfn",      # Saldo da carteira de credito total, R$ milhoes (ex-"spread_pj")
    20541: "credito_pj_saldo",       # Saldo da carteira de credito PJ, R$ milhoes
}

macro, falhas = None, []
for codigo, nome in SERIES.items():
    print(f"Baixando {nome:24s} (SGS {codigo})...", end=" ", flush=True)
    try:
        serie = buscar_sgs(codigo, nome)
        macro = serie if macro is None else macro.merge(serie, on="data", how="outer")
        print(f"ok ({len(serie)} obs)")
    except Exception as e:
        falhas.append(nome)
        print(f"ERRO: {str(e)[:70]}")

macro = macro.sort_values("data").reset_index(drop=True)

print(f"\nShape: {macro.shape}  |  periodo: {macro['data'].min().date()} -> {macro['data'].max().date()}")
if falhas:
    print(f"Series que falharam: {falhas}")
print("\nNulos por coluna:")
print(macro.isnull().sum().to_string())
print("\nAmostra:")
print(macro.head(3).to_string(index=False))

In [ ]:
# Grava macro.parquet.
# - Se o arquivo nao existe: grava.
# - Se existe mas o ESQUEMA mudou (colunas novas ou renomeadas — revisao A4c): regrava.
# - Se existe com o mesmo esquema: NAO sobrescreve; compara com o que acabou de ser baixado
#   (verificacao de reprodutibilidade). O BCB revisa observacoes recentes, entao pequenas
#   diferencas na cauda sao esperadas.
destino = PROC / "macro.parquet"

if destino.exists():
    antigo = pd.read_parquet(destino)
    faltam = [c for c in macro.columns if c not in antigo.columns]
    if faltam:
        macro.to_parquet(destino, index=False)
        print(f"Arquivo existia sem as colunas {faltam} -> regravado com o esquema novo "
              f"({macro.shape[0]} linhas x {macro.shape[1]} colunas)")
    else:
        comuns = [c for c in macro.columns if c in antigo.columns]
        cmp = (antigo[comuns].set_index("data")
               .join(macro[comuns].set_index("data"),
                     lsuffix="_disco", rsuffix="_novo", how="inner"))
        print(f"Arquivo ja existe -> nao sobrescrito. Comparando {len(cmp)} meses em comum:\n")
        for c in [x for x in comuns if x != "data"]:
            a, b = cmp[f"{c}_disco"], cmp[f"{c}_novo"]
            both = a.notna() & b.notna()
            difmax = (a[both] - b[both]).abs().max() if both.any() else float("nan")
            status = "identico" if (both.any() and difmax < 1e-9) else f"dif. max = {difmax:.4g}"
            print(f"  {c:24s} {status}")
        print("\nPara regerar do zero, apague data/processed/macro.parquet e reexecute.")
else:
    macro.to_parquet(destino, index=False)
    print(f"Salvo: {destino}  ({macro.shape[0]} linhas x {macro.shape[1]} colunas)")